In [0]:
# Create widgets
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "Source File Format")

# Get widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
landing_path = f"/Volumes/adb_caresync_smd/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

try:
    files = dbutils.fs.ls(landing_path)
    if not files:
        print(f"No new files received at: {landing_path}")
        dbutils.notebook.exit("No new files received")
except Exception:
    print(f"No new files received at: {landing_path}")
    dbutils.notebook.exit("No new files received")

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

In [0]:
from pyspark.sql.functions import lit, current_timestamp

updated_df = landing_df.withColumn("landing_timestamp", lit(landing_timestamp)) \
                       .withColumn("insert_timestamp", current_timestamp())

In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_caresync_smd.bronze.{table_name}")